In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
source_table = "retail_project.silver.sales_cleaned"
target_table = "retail_project.gold.dim_customers"
silver_df = spark.table(source_table)

In [0]:
new_customer_data = (
    silver_df.select("cust_id", "city", "region").distinct()
    .withColumn("customer_key", F.sha2(F.col("cust_id"), 256))
    .withColumn("is_current", F.lit(True))
    .withColumn("start_date", F.current_timestamp())
    .withColumn("end_date", F.lit(None).cast("timestamp"))                 
)

In [0]:
def upsert_customer_dimension(df, target_table):

    if not spark.catalog.tableExists(target_table):
        df.write.format("delta").saveAsTable(target_table)
    else:
        target_dt = DeltaTable.forName(spark, target_table)

        target_dt.alias("t").merge(
            df.alias("s"),
            "t.cust_id = s.cust_id AND t.is_current = True"
        ).whenMatchedUpdate(
            condition = "t.city != s.city OR t.region != s.region",
            set = {"is_current": "False", "end_date": "current_timestamp()"}
        ).execute()

        df.write.format("delta").mode("append").saveAsTable(target_table)


In [0]:
upsert_customer_dimension(new_customer_data, target_table)